In [5]:
# train_conveyor_model.py

import os
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder


DATA_PATH = "/home/santhosh/AMDA/datasets/CONVEYOR_dataset.csv"
MODEL_DIR = "/home/santhosh/AMDA/models/CONVEYOR"
MODEL_PATH = f"{MODEL_DIR}/CONVEYOR_failure_model.pkl"

FEATURE_COLS = [
    "vibration_mm_s",
    "motor_current_A",
    "gearbox_temperature_C",
    "belt_speed_mps",
    "load_kg",
    "belt_alignment_mm",
]

CLIP_RANGES = {
    "vibration_mm_s": (0.1, 12),
    "motor_current_A": (1, 30),
    "gearbox_temperature_C": (20, 130),
    "belt_speed_mps": (0, 5),
    "load_kg": (0, 650),
    "belt_alignment_mm": (0, 20),
}


def add_realistic_noise(df, noise_level=0.06):
    np.random.seed(42)
    df = df.copy()

    for col in FEATURE_COLS:
        df[col] += df[col] * np.random.uniform(-noise_level, noise_level, len(df))
        df[col] += np.random.normal(0, df[col].std() * 0.03, len(df))
        low, high = CLIP_RANGES[col]
        df[col] = df[col].clip(low, high)

    return df


def train():
    os.makedirs(MODEL_DIR, exist_ok=True)

    df = pd.read_csv(DATA_PATH)
    df = df.dropna(subset=FEATURE_COLS + ["active_failure"])

    print("Class distribution:")
    print(df["active_failure"].value_counts())

    df = add_realistic_noise(df)

    X = df[FEATURE_COLS]
    y = df["active_failure"]

    encoder = LabelEncoder()
    y_encoded = encoder.fit_transform(y)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
    )

    model = RandomForestClassifier(
        n_estimators=400,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
    print("Weighted F1:", f1_score(y_test, y_pred, average="weighted"))

    print(classification_report(y_test, y_pred, target_names=encoder.classes_))
    print(confusion_matrix(y_test, y_pred))

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X, y_encoded, cv=cv, scoring="f1_macro", n_jobs=-1)
    print("CV Macro F1:", cv_scores.mean())

    importance = pd.DataFrame({
        "feature": FEATURE_COLS,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False)

    importance.to_csv(f"{MODEL_DIR}/CONVEYOR_feature_importance.csv", index=False)

    plt.figure(figsize=(8, 5))
    plt.barh(importance["feature"], importance["importance"])
    plt.gca().invert_yaxis()
    plt.title("Conveyor Feature Importance")
    plt.tight_layout()
    plt.savefig(f"{MODEL_DIR}/CONVEYOR_feature_importance.png")
    plt.close()

    joblib.dump({
        "model": model,
        "label_encoder": encoder,
        "feature_cols": FEATURE_COLS,
        "classes": list(encoder.classes_),
        "machine_type": "conveyor",
        "machine_id": "CONVEYOR_04",
    }, MODEL_PATH)

    print("Saved:", MODEL_PATH)


if __name__ == "__main__":
    train()

Class distribution:
active_failure
none                      1024
overload                  1010
jam                        996
belt_misalignment          995
belt_slip                  993
roller_bearing_failure     992
gearbox_overheating        990
Name: count, dtype: int64
Accuracy: 0.9066666666666666
Macro F1: 0.9065132693423058
Weighted F1: 0.9066843107552185
                        precision    recall  f1-score   support

     belt_misalignment       0.90      0.84      0.86       298
             belt_slip       0.91      0.92      0.91       298
   gearbox_overheating       0.89      0.88      0.89       297
                   jam       0.97      0.96      0.96       299
                  none       0.97      0.95      0.96       307
              overload       0.87      0.87      0.87       303
roller_bearing_failure       0.85      0.92      0.88       298

              accuracy                           0.91      2100
             macro avg       0.91      0.91      0.91 